In [1]:
import os
from openai import OpenAI
from tqdm import tqdm
from collections import Counter

##############################################
# OpenAI Client
##############################################

client = OpenAI(api_key=os.getenv("OPEN_API_KEY"))

##############################################
# 1 Document Loading
##############################################

def load_docs(folder_path):

    documents = []

    for file in os.listdir(folder_path):

        if file.endswith((".puml", ".plantuml")):

            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:

                content = f.read()

                documents.append({
                    "id": file,
                    "raw": content
                })

    return documents


##############################################
# 2 LLM Reasoning (Direct)
##############################################

def llm_reasoning(query_raw):

    prompt = f"""

Identify the GoF design pattern used in the following PlantUML diagram.

PLANTUML DIAGRAM:
{query_raw}

Respond ONLY with the design pattern name.
"""

    response = client.chat.completions.create(

        model="gpt-5.1",

        messages=[
            {"role": "system", "content": "You are an expert in software modeling and GoF design patterns."},
            {"role": "user", "content": prompt}
        ],

    )

    return response.choices[0].message.content.strip()


##############################################
# 3 Pattern Detection
##############################################

def detect_pattern(test_doc):

    prediction = llm_reasoning(test_doc["raw"])

    return prediction


##############################################
# 4 Evaluation
##############################################

def evaluate_detection(test_folder):

    test_docs = load_docs(test_folder)

    correct_counts = Counter()
    total_counts = Counter()

    for doc in tqdm(test_docs):

        true_label = doc["id"].split('-')[0].split('_')[0]

        total_counts[true_label] += 1

        predicted_label = detect_pattern(doc)

        if predicted_label.lower() == true_label.lower():
            correct_counts[true_label] += 1

        print(f"File: {doc['id']} | True: {true_label} | Pred: {predicted_label}")

    per_pattern_accuracy = {}

    for label in total_counts:
        per_pattern_accuracy[label] = correct_counts[label] / total_counts[label]

    overall_accuracy = sum(correct_counts.values()) / sum(total_counts.values())

    print("\nPer-Pattern Accuracy:")

    for label, acc in per_pattern_accuracy.items():
        print(f"{label}: {acc:.4f}")

    print(f"\nOverall Accuracy: {overall_accuracy:.4f}")

    return overall_accuracy, per_pattern_accuracy

In [2]:
test_folder = "PMART_PUML/test"

overall_acc, per_pattern_acc = evaluate_detection(test_folder)

print("\nFinished evaluation.")

  1%|          | 1/83 [00:01<01:35,  1.16s/it]

File: Adapter-430.puml | True: Adapter | Pred: Adapter


  2%|▏         | 2/83 [00:02<01:22,  1.02s/it]

File: Template Method-2222.puml | True: Template Method | Pred: Factory Method


  4%|▎         | 3/83 [00:03<01:31,  1.14s/it]

...

File: Proxy-2040.puml | True: Proxy | Pred: Proxy


 99%|█████████▉| 82/83 [01:44<00:01,  1.14s/it]

File: Abstract Factory-42.puml | True: Abstract Factory | Pred: Factory Method


100%|██████████| 83/83 [01:45<00:00,  1.27s/it]

File: Adapter-52.puml | True: Adapter | Pred: Observer

Per-Pattern Accuracy:
Adapter: 0.4286
Template Method: 0.4444
Abstract Factory: 0.2500
Singleton: 0.2727
Strategy: 0.5000
State: 0.3333
Iterator: 0.0000
Command: 0.8000
Visitor: 1.0000
Proxy: 1.0000
Factory Method: 0.0000

Overall Accuracy: 0.3735

Finished evaluation.
